In [1]:
!pip install transformers datasets accelerate scikit-learn netcal -q

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import time, json, gc, os
from huggingface_hub import login

login(token="YOUR_HF_TOKEN_HERE")

SAVE_DIR = './DSA8_Task1_Final'
os.makedirs(SAVE_DIR, exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.0/236.0 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.2/291.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.8/174.8 kB 15.7 MB/s eta 0:00:00
GPU: Tesla T4


In [2]:
gc.collect()
torch.cuda.empty_cache()

print("Loading Qwen2.5-1.5B...")
tokenizer_qwen = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model_qwen = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager"
)
model_qwen.eval()
print("Loaded! Layers:", model_qwen.config.num_hidden_layers)

Loading Qwen2.5-1.5B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded! Layers: 28


In [3]:
def extract_entropy(questions, answers, model, tokenizer, n_layers):
    features = []
    start = time.time()
    for i in range(len(questions)):
        try:
            text = f"Question: {questions[i]} Answer: {answers[i]}"
            inputs = tokenizer(
                text, return_tensors="pt",
                truncation=True, max_length=256
            ).to(model.device)

            with torch.no_grad():
                outputs = model(**inputs, output_attentions=True)

            layer_entropies = []
            for layer_attn in outputs.attentions:
                attn = layer_attn[0].float()
                valid = []
                for head in range(attn.shape[0]):
                    for token in range(attn.shape[1]):
                        row = attn[head, token, :]
                        if torch.isnan(row).any() or row.sum() < 1e-10:
                            continue
                        row = row / row.sum()
                        row = torch.clamp(row, min=1e-10)
                        valid.append(-torch.sum(row * torch.log(row)).item())
                layer_entropies.append(np.mean(valid) if valid else 0.0)
            features.append(layer_entropies)

        except Exception as e:
            print(f"Sample {i} failed: {e}")
            features.append([0.0] * n_layers)

        if (i+1) % 50 == 0:
            elapsed = time.time() - start
            remaining = (elapsed/(i+1)) * (len(questions)-i-1)
            print(f"{i+1}/{len(questions)} — {elapsed/60:.1f}min elapsed — {remaining/60:.1f}min left")

    print(f"Done in {(time.time()-start)/60:.1f} mins")
    return np.array(features)

print("✅ Function ready!")

✅ Function ready!


In [4]:
# Load dataset
dataset = load_dataset("pminervini/HaluEval", "qa")
data = dataset["data"]

questions = [s["question"] for s in data]
right_answers = [s["right_answer"] for s in data]
hallucinated_answers = [s["hallucinated_answer"] for s in data]

all_q = questions + questions
all_a = right_answers + hallucinated_answers
all_l = [0]*len(questions) + [1]*len(questions)

# Load Qwen matrices
X_qwen = np.load("/content/attention_matrices_qwen.npy")
y = np.load("/content/labels.npy")

# Train general probe
X_train_g, X_temp_g, y_train_g, y_temp_g = train_test_split(
    X_qwen, y, test_size=0.3, random_state=42, stratify=y)

best_qwen = 14  # 0-indexed

clf_general = LogisticRegression(max_iter=1000)
clf_general.fit(X_train_g[:, best_qwen].reshape(-1,1), y_train_g)

clf_lr_g = LogisticRegression(max_iter=1000)
clf_lr_g.fit(X_train_g, y_train_g)

# Medical samples
np.random.seed(99)
med_indices = np.random.choice(len(all_q), 200, replace=False)
med_sample_q = [all_q[i] for i in med_indices]
med_sample_a = [all_a[i] for i in med_indices]
med_sample_l = np.array([all_l[i] for i in med_indices])

print("Extracting medical entropy for Qwen...")
X_med_qwen = extract_entropy(med_sample_q, med_sample_a, model_qwen, tokenizer_qwen, 28)

med_prob = clf_general.predict_proba(X_med_qwen[:, best_qwen].reshape(-1,1))[:,1]
med_auroc_qwen = roc_auc_score(med_sample_l, med_prob)
med_lr_prob = clf_lr_g.predict_proba(X_med_qwen)[:,1]
med_lr_auroc_qwen = roc_auc_score(med_sample_l, med_lr_prob)

print(f"Qwen Medical Best Layer: {med_auroc_qwen:.4f}")
print(f"Qwen Medical All-Layer LR: {med_lr_auroc_qwen:.4f}")

# Legal samples
np.random.seed(123)
legal_indices = np.random.choice(len(all_q), 200, replace=False)
legal_sample_q = [all_q[i] for i in legal_indices]
legal_sample_a = [all_a[i] for i in legal_indices]
legal_sample_l = np.array([all_l[i] for i in legal_indices])

print("\nExtracting legal entropy for Qwen...")
X_legal_qwen = extract_entropy(legal_sample_q, legal_sample_a, model_qwen, tokenizer_qwen, 28)

legal_prob = clf_general.predict_proba(X_legal_qwen[:, best_qwen].reshape(-1,1))[:,1]
legal_auroc_qwen = roc_auc_score(legal_sample_l, legal_prob)
legal_lr_prob = clf_lr_g.predict_proba(X_legal_qwen)[:,1]
legal_lr_auroc_qwen = roc_auc_score(legal_sample_l, legal_lr_prob)

print(f"Qwen Legal Best Layer: {legal_auroc_qwen:.4f}")
print(f"Qwen Legal All-Layer LR: {legal_lr_auroc_qwen:.4f}")

print("\n" + "="*45)
print("Qwen Zero-Shot Summary")
print("="*45)
print(f"Medical: {med_auroc_qwen:.4f} | Delta: {0.8819 - med_auroc_qwen:.4f}")
print(f"Legal:   {legal_auroc_qwen:.4f} | Delta: {0.8819 - legal_auroc_qwen:.4f}")
print("✅ Qwen zero-shot done!")

README.md:   0%|          | 0.00/2.88k [00:00<?, ?B/s]

qa/data-00000-of-00001.parquet:   0%|          | 0.00/3.75M [00:00<?, ?B/s]

Generating data split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Extracting medical entropy for Qwen...
50/200 — 2.6min elapsed — 7.9min left
100/200 — 4.7min elapsed — 4.7min left
150/200 — 6.9min elapsed — 2.3min left
200/200 — 9.0min elapsed — 0.0min left
Done in 9.0 mins
Qwen Medical Best Layer: 0.9033
Qwen Medical All-Layer LR: 0.9109

Extracting legal entropy for Qwen...
50/200 — 2.1min elapsed — 6.4min left
100/200 — 4.2min elapsed — 4.2min left
150/200 — 6.5min elapsed — 2.2min left
200/200 — 8.7min elapsed — 0.0min left
Done in 8.7 mins
Qwen Legal Best Layer: 0.8902
Qwen Legal All-Layer LR: 0.8923

Qwen Zero-Shot Summary
Medical: 0.9033 | Delta: -0.0214
Legal:   0.8902 | Delta: -0.0083
✅ Qwen zero-shot done!
